# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
# import yaml
import json
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.llms.base import TextGenerator
from os.path import join

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# llm_config_path = "../../config/llm_config.yml"
# llm_config = yaml.safe_load(open(llm_config_path))
# llm_config["provider"] = llm_provider
# llm_config["model"] = llm_model

## Extract Features X Used in Model

In [3]:
# create dict to store features
features = {}

In [4]:
multirun_analyses['analyses']['0'].keys()

dict_keys(['cvars', 'transform_code', 'm_code'])

In [5]:
def get_feature_transforms(llm_assistant: TextGenerator, transform_code: str,
                           feature_columns: list[str],
                           feature_description: str):
    """
    Given a list of feature columns, check if the columns are transformed in the
    transform code and return the code that performs the transformation.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        transform_code (str): The code that performs the transformations.
        feature_columns (list[str]): The list of feature columns to check.
        
    Returns:
        dict: A dictionary of feature columns and the code that performs the transformation.
    """
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        performing data cleaning and preprocessing tasks."""
    transform_responses = []
    for feature_column in feature_columns:
        find_transform_prompt = f"""Given the following code:
            <Code>
            {transform_code}
            </Code>
            and the feature column:
            <Feature Column>
            {feature_column}
            </Feature Column>
            with description:
            <Feature Description>
            {feature_description}
            </Feature Description>
            determine if the column is transformed in the code. \
            If it is, return only the corresponding lines of code that perform the transformation. \
            If it is not, return "No transformation code found."
            """
        response = llm_assistant.generate([{"role": "system",
                                            "content": system_prompt},
                                           {"role": "user",
                                            "content": find_transform_prompt}])
        transform_responses.append(response)
    return transform_responses
                

In [6]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # get the features from each analysis
    # this should include the independent and control variables
    ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
    control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in ind_vars and control_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    for dict_idx, var in enumerate(ind_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
    for dict_idx, var in enumerate(control_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
    

[2025-11-11 12:28:24.10][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-11 12:28:24.43][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [7]:
ind_vars

[{'description': 'Continuous masculinity-femininity index for the hurricane name, standardized (z-score). Higher values = more feminine name. Primary independent variable capturing perceived femininity of the hurricane name.',
  'columns': ['masfem_z'],
  'transform_code': ["if 'masfem' in df.columns:\n    mean_m = df['masfem'].mean()\n    std_m = df['masfem'].std(ddof=0)\n    df['masfem_z'] = (df['masfem'] - mean_m) / (std_m if std_m != 0 else 1.0)\nelse:\n    df['masfem_z'] = np.nan"]},
 {'description': 'Alternative continuous masculinity-femininity index obtained from MTurk raters, standardized (z-score). Used as robustness for the name femininity measure.',
  'columns': ['masfem_mturk_z'],
  'transform_code': ["if 'masfem_mturk' in df.columns:\n    mean_mt = df['masfem_mturk'].mean()\n    std_mt = df['masfem_mturk'].std(ddof=0)\n    df['masfem_mturk_z'] = (df['masfem_mturk'] - mean_mt) / (std_mt if std_mt != 0 else 1.0)\nelse:\n    df['masfem_mturk_z'] = np.nan"]},
 {'description':

In [8]:
control_vars

[{'description': 'Maximum sustained wind speed at landfall (NOAA). Controls for physical severity of the storm.',
  'is_moderator': False,
  'moderator_on': None,
  'columns': ['wind'],
  'transform_code': ["num_cols = ['masfem', 'masfem_mturk', 'gender_mf', 'alldeaths', 'ndam15', 'wind', 'category', 'min', 'year', 'elapsedyrs']\nfor c in num_cols:\n    if c in df.columns:\n        df[c] = pd.to_numeric(df[c], errors='coerce')"]},
 {'description': 'Saffir-Simpson category (1-5). Controls for storm intensity.',
  'is_moderator': False,
  'moderator_on': None,
  'columns': ['category'],
  'transform_code': ["num_cols = ['masfem', 'masfem_mturk', 'gender_mf', 'alldeaths', 'ndam15', 'wind', 'category', 'min', 'year', 'elapsedyrs']\nfor c in num_cols:\n    if c in df.columns:\n        df[c] = pd.to_numeric(df[c], errors='coerce')"]},
 {'description': 'Minimum central pressure at landfall. Controls for storm intensity (lower = stronger).',
  'is_moderator': False,
  'moderator_on': None,
  '